In [1]:
# ============================================================
# PMSM THERMAL PARAMETER IDENTIFICATION
# STEP 1: PHYSICAL CONSISTENCY + IDENTIFIABILITY ANALYSIS
# ============================================================

import os
import numpy as np
import pandas as pd

from scipy.optimize import lsq_linear
from numpy.linalg import cond, matrix_rank

# ============================================================
# CONFIGURATION
# ============================================================

DATA_PATH = "/Users/prashantsingh.basnet@iqvia.com/Documents/sim/.venv/measures_v2.csv"

# ------------------------------------------------------------
# Required columns
# ------------------------------------------------------------

REQUIRED_COLUMNS = [
    "u_q",
    "coolant",
    "stator_winding",
    "u_d",
    "stator_tooth",
    "motor_speed",
    "i_d",
    "i_q",
    "pm",
    "stator_yoke",
    "ambient",
    "torque",
    "profile_id"
]

# ============================================================
# LOAD DATA
# ============================================================

print("=" * 70)
print("PMSM PHYSICAL CONSISTENCY / PARAMETER IDENTIFICATION")
print("=" * 70)

print("\nLoading dataset...")

df = pd.read_csv(DATA_PATH)

print(f"Rows    : {len(df):,}")
print(f"Columns : {len(df.columns)}")

missing = [
    c for c in REQUIRED_COLUMNS
    if c not in df.columns
]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

print("\nRequired columns verified.")

# ============================================================
# BASIC CLEANING
# ============================================================

df = df[REQUIRED_COLUMNS].copy()

df = df.replace(
    [np.inf, -np.inf],
    np.nan
)

before = len(df)

df = df.dropna()

after = len(df)

print(f"\nRemoved NaN/Inf rows: {before - after:,}")
print(f"Remaining rows     : {after:,}")

# ============================================================
# SORT BY PROFILE
# ============================================================

df = df.sort_values(
    ["profile_id"]
).reset_index(drop=True)

profiles = df["profile_id"].unique()

print(f"\nProfiles: {len(profiles)}")

# ============================================================
# PHYSICAL CONSTANTS
# ============================================================

# IMPORTANT:
# These are placeholders ONLY.
# Do NOT use them as final paper parameters.
#
# Replace them with experimentally documented motor values
# when available.

R_S0 = 0.05       # Ohm
T0 = 20.0         # deg C
ALPHA = 0.00393   # 1/deg C

# ============================================================
# ELECTRICAL / MECHANICAL POWER
# ============================================================

print("\nCalculating power quantities...")

# Conventional dq power equation
df["P_electrical"] = (
    1.5 *
    (
        df["u_d"] * df["i_d"] +
        df["u_q"] * df["i_q"]
    )
)

# Dataset motor_speed is assumed to be rpm
df["omega_m"] = (
    2.0 *
    np.pi *
    df["motor_speed"] /
    60.0
)

# Mechanical output power
df["P_mechanical"] = (
    df["torque"] *
    df["omega_m"]
)

# Electrical-mechanical power difference
df["P_loss_total"] = (
    df["P_electrical"]
    -
    df["P_mechanical"]
)

# ============================================================
# TEMPERATURE-DEPENDENT STATOR RESISTANCE
# ============================================================

df["R_s"] = (
    R_S0 *
    (
        1.0 +
        ALPHA *
        (
            df["stator_winding"] - T0
        )
    )
)

# ============================================================
# COPPER LOSS
# ============================================================

df["P_copper"] = (
    1.5 *
    df["R_s"] *
    (
        df["i_d"] ** 2 +
        df["i_q"] ** 2
    )
)

# Remaining loss after copper loss
df["P_remaining"] = (
    df["P_loss_total"]
    -
    df["P_copper"]
)

# ============================================================
# BASIC POWER STATISTICS
# ============================================================

print("\n" + "=" * 70)
print("POWER CONSISTENCY")
print("=" * 70)

power_columns = [
    "P_electrical",
    "P_mechanical",
    "P_loss_total",
    "P_copper",
    "P_remaining"
]

print(
    df[power_columns]
    .describe()
    .T
)

# ============================================================
# NEGATIVE POWER FRACTIONS
# ============================================================

print("\nNegative-value diagnostics:")

for col in power_columns:

    negative_fraction = (
        (df[col] < 0).mean()
    )

    print(
        f"{col:20s} : "
        f"{negative_fraction * 100:.3f}% negative"
    )

# ============================================================
# PROFILE-LEVEL STATISTICS
# ============================================================

profile_summary = (
    df.groupby("profile_id")
    .agg(
        samples=("profile_id", "size"),

        mean_Pe=("P_electrical", "mean"),
        mean_Pm=("P_mechanical", "mean"),
        mean_Ploss=("P_loss_total", "mean"),
        mean_Pcu=("P_copper", "mean"),
        mean_Premaining=("P_remaining", "mean"),

        mean_Tw=("stator_winding", "mean"),
        mean_Tt=("stator_tooth", "mean"),
        mean_Ty=("stator_yoke", "mean"),
        mean_Tpm=("pm", "mean"),
        mean_Tc=("coolant", "mean"),
        mean_Ta=("ambient", "mean")
    )
)

print("\n" + "=" * 70)
print("PROFILE-LEVEL SUMMARY")
print("=" * 70)

print(
    profile_summary.describe().T
)

# ============================================================
# SAMPLING INTERVAL
# ============================================================

print("\n" + "=" * 70)
print("SAMPLING INTERVAL")
print("=" * 70)

print(
    """
The dataset does not contain an explicit timestamp column
in the provided schema.

Therefore Δt cannot be mathematically inferred from the
CSV unless the dataset documentation specifies the sampling
frequency.

DO NOT assume Δt until this is verified.
"""
)

# ============================================================
# TEMPERATURE DIFFERENCES
# ============================================================

df["dTw"] = (
    df["stator_winding"]
    - df["stator_tooth"]
)

df["dTt_y"] = (
    df["stator_tooth"]
    - df["stator_yoke"]
)

df["dTt_pm"] = (
    df["stator_tooth"]
    - df["pm"]
)

df["dTy_c"] = (
    df["stator_yoke"]
    - df["coolant"]
)

df["dTy_a"] = (
    df["stator_yoke"]
    - df["ambient"]
)

temperature_differences = [
    "dTw",
    "dTt_y",
    "dTt_pm",
    "dTy_c",
    "dTy_a"
]

print("\n" + "=" * 70)
print("THERMAL DRIVING-FORCE STATISTICS")
print("=" * 70)

print(
    df[temperature_differences]
    .describe()
    .T
)

# ============================================================
# IDENTIFIABILITY PREPARATION
# ============================================================

print("\n" + "=" * 70)
print("THERMAL PARAMETER IDENTIFIABILITY")
print("=" * 70)

print(
    """
We now construct the regressions required to identify
thermal capacitances and conductances.

The fundamental equation is:

C_i * dT_i/dt =
P_i + Σ G_ij(T_j - T_i)

However, Δt is not yet known from the CSV.

Therefore we initially construct the DESIGN VARIABLES
without multiplying by an assumed Δt.

Once Δt is verified, these equations will be solved.
"""
)

# ============================================================
# DISPLAY DESIGN MATRIX CONDITIONING
# ============================================================

# We can examine whether the thermal driving forces
# themselves contain enough independent variation.

thermal_X = df[
    [
        "dTw",
        "dTt_y",
        "dTt_pm",
        "dTy_c",
        "dTy_a"
    ]
].to_numpy(
    dtype=np.float64
)

# Remove non-finite rows
thermal_X = thermal_X[
    np.all(
        np.isfinite(thermal_X),
        axis=1
    )
]

print(
    f"\nThermal design matrix shape: "
    f"{thermal_X.shape}"
)

rank = matrix_rank(thermal_X)

print(
    f"Matrix rank: {rank} / {thermal_X.shape[1]}"
)

try:

    condition_number = cond(
        thermal_X
    )

    print(
        f"Condition number: "
        f"{condition_number:.6e}"
    )

except Exception as e:

    print(
        "Condition number could not be calculated:",
        e
    )

# ============================================================
# CORRELATION ANALYSIS
# ============================================================

print("\n" + "=" * 70)
print("THERMAL VARIABLE CORRELATION")
print("=" * 70)

corr = df[
    [
        "stator_winding",
        "stator_tooth",
        "stator_yoke",
        "pm",
        "coolant",
        "ambient",
        "dTw",
        "dTt_y",
        "dTt_pm",
        "dTy_c",
        "dTy_a",
        "P_copper",
        "P_remaining"
    ]
].corr()

pd.set_option(
    "display.max_columns",
    None
)

print(corr.round(4))

# ============================================================
# PROFILE-LEVEL THERMAL RANGE
# ============================================================

print("\n" + "=" * 70)
print("THERMAL EXCITATION BY PROFILE")
print("=" * 70)

thermal_range = (
    df.groupby("profile_id")
    .agg(
        Tw_range=(
            "stator_winding",
            lambda x: x.max() - x.min()
        ),

        Tt_range=(
            "stator_tooth",
            lambda x: x.max() - x.min()
        ),

        Ty_range=(
            "stator_yoke",
            lambda x: x.max() - x.min()
        ),

        Tpm_range=(
            "pm",
            lambda x: x.max() - x.min()
        ),

        Pcu_range=(
            "P_copper",
            lambda x: x.max() - x.min()
        )
    )
)

print(
    thermal_range.describe().T
)

# ============================================================
# SAVE DIAGNOSTIC RESULTS
# ============================================================

OUTPUT_DIR = "/Users/prashantsingh.basnet@iqvia.com/Documents/sim/.venv/physics_diagnostics"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

profile_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "profile_power_summary.csv"
    )
)

thermal_range.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "thermal_excitation_by_profile.csv"
    )

)

corr.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "thermal_correlation.csv"
    )
)

print("\n" + "=" * 70)
print("DIAGNOSTICS COMPLETE")
print("=" * 70)

print(
    f"\nResults saved to:"
    f"\n{OUTPUT_DIR}"
)

print(
    """
NEXT:
Paste the COMPLETE OUTPUT of this script.

Do NOT train the model yet.

We will use the output to determine:

1. Whether the power equations are consistent.
2. Whether copper loss is plausible.
3. Whether the remaining loss is physically usable.
4. Whether the thermal states have sufficient excitation.
5. Whether the thermal parameters are identifiable.
6. Whether our four-node thermal model should be
   retained, simplified, or expanded.
"""
)

PMSM PHYSICAL CONSISTENCY / PARAMETER IDENTIFICATION

Loading dataset...
Rows    : 1,330,816
Columns : 13

Required columns verified.

Removed NaN/Inf rows: 0
Remaining rows     : 1,330,816

Profiles: 69

Calculating power quantities...

POWER CONSISTENCY
                  count         mean           std           min          25%  \
P_electrical  1330816.0  7630.838262  18583.042287 -4.651746e+04    -4.693924   
P_mechanical  1330816.0  6513.603236  18580.971923 -5.526779e+04   -26.937972   
P_loss_total  1330816.0  1117.235026   1221.295319 -3.147552e+04    98.135416   
P_copper      1330816.0  1767.384650   2107.958345  1.162785e-10    12.612000   
P_remaining   1330816.0  -650.149623   1402.834858 -3.148042e+04 -1052.191423   

                      50%           75%           max  
P_electrical  1457.590478  21225.461869  52471.639499  
P_mechanical  1027.488385  20439.095479  49618.186920  
P_loss_total   765.550453   1652.311217  28427.112130  
P_copper      1121.233257   2732.